In [0]:
# ============================================================
# SILVER TRANSFORMATION
# price_and_demand
#
# Purpose:
# Clean and standardize electricity market data
# ============================================================

In [0]:
%run "/Workspace/Users/vtilakumara@academic.mit.edu.au/au-energy-data-pipeline/config"

In [0]:
# if the job parameter is not set, default to dev
try:
    ENV = dbutils.widgets.get("ENV")
except:
    ENV = "dev"

print(f"Starting pipeline for {ENV}")
print(f"Environment is {ENV}")
print(f"Starting Task 2 - Silver layer creation for price and demand")

In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS SILVER_TABLE;

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

display(bronze_df.limit(10))

In [0]:
bronze_df.printSchema()

In [0]:
# Rename columns 
renamed_df = bronze_df.select(
    F.col("REGION").alias("region"),
    F.col("SETTLEMENTDATE").alias("settlement_raw"),
    F.col("TOTALDEMAND").alias("total_demand_raw"),
    F.col("RRP").alias("price_rrp_raw"),
    F.col("PERIODTYPE").alias("period_type"),
    F.col("source_file"),
    F.col("ingested_at")
)

display(renamed_df)

In [0]:
# Selects and transforms columns from renamed_df:
# - Converts settlement_raw to timestamp as settlement_timestamp
# - Casts total_demand_raw and price_rrp_raw to double
# - Renames columns for clarity
typed_df = renamed_df.select(
    F.col("region"),
    F.to_timestamp("settlement_raw", "yyyy/MM/dd HH:mm:ss").alias("settlement_timestamp"),
    F.col("total_demand_raw").cast("double").alias("total_demand_mw"),
    F.col("price_rrp_raw").cast("double").alias("price_rrp"),
    F.col("period_type"),
    F.col("source_file"),
    F.col("ingested_at")
)

typed_df.printSchema()


In [0]:
# Add columns for year, month, day, hour, minute, day_of_week
enriched_df = typed_df \
.withColumn("year", F.year("settlement_timestamp")) \
.withColumn("month", F.month("settlement_timestamp")) \
.withColumn("day", F.dayofmonth("settlement_timestamp")) \
.withColumn("hour", F.hour("settlement_timestamp")) \
.withColumn("minute", F.minute("settlement_timestamp")) \
.withColumn("day_of_week", F.date_format("settlement_timestamp", "E"))

display(enriched_df)

In [0]:
valid_regions = ["VIC1","NSW1","QLD1","SA1","TAS1"]

In [0]:
print("Checking Data Quality...")
# Adds a "quality_status" column to enriched_df based on data quality checks:
# - INVALID_TIMESTAMP: settlement_timestamp is null
# - INVALID_REGION: region is not in valid_regions
# - MISSING_DEMAND: total_demand_mw is null
# - NEGATIVE_DEMAND: total_demand_mw is negative
# - VALID: passes all checks

quality_df = enriched_df.withColumn(
    "quality_status",    
    F.when(F.col("settlement_timestamp").isNull(), "INVALID_TIMESTAMP")    
    .when(~F.col("region").isin(valid_regions), "INVALID_REGION")    
    .when(F.col("total_demand_mw").isNull(), "MISSING_DEMAND")    
    .when(F.col("total_demand_mw") < 0, "NEGATIVE_DEMAND")    
    .otherwise("VALID")
)

In [0]:
display(quality_df)

In [0]:
# Filter quality_df to get valid_df and invalid_df
valid_df = quality_df.filter(F.col("quality_status") == "VALID")

invalid_df = quality_df.filter(F.col("quality_status") != "VALID")

In [0]:
# Deduplicate valid_df based on region and settlement_timestamp
dedup_df = valid_df.dropDuplicates(["region","settlement_timestamp"])

In [0]:
# Create Silver Table
silver_df = dedup_df.select(
    "region",
    "settlement_timestamp",
    "total_demand_mw",
    "price_rrp",
    "period_type",
    "year",
    "month",
    "day",
    "hour",
    "minute",
    "day_of_week",
    "source_file",
    "ingested_at",
    "quality_status"
)

display(silver_df)

In [0]:
(
silver_df.write
.format("delta")
.mode("overwrite")
.saveAsTable(SILVER_TABLE)
)

In [0]:
(
invalid_df.write
.format("delta")
.mode("overwrite")
.saveAsTable(REJECT_TABLE)
)

In [0]:
display(spark.sql(f"SELECT * FROM {SILVER_TABLE} LIMIT 10"))

In [0]:
%sql
-- Validation Checks
-- Check row count
SELECT COUNT(*) 
FROM energyau2026_7405617942738947.energyau_dev_silver.price_and_demand
WHERE quality_status = 'VALID';

In [0]:
%sql
-- Check duplicates
SELECT region, settlement_timestamp, COUNT(*)
FROM energyau2026.energyau_dev_silver.price_and_demand
GROUP BY region, settlement_timestamp
HAVING COUNT(*) > 1

In [0]:
%sql
-- Check quality distribution
SELECT quality_status, COUNT(*)
FROM energyau2026.energyau_dev_silver.price_and_demand
GROUP BY quality_status